# BTCUSD Multi-Timeframe Transformer Training (Standalone)

Delta Exchange India data → feature engineering → market-understanding labels → Transformer → ONNX export.

Upload **this notebook only** to Google Colab — no GitHub clone, no file uploads. Historical
OHLCV, funding, and OI are pulled from the **Delta Exchange India public API** at runtime.

**Run all cells top-to-bottom.** Training code is inline in this notebook (readable Python cells).

### Cell map (troubleshooting)

| Section | What to inspect |
|---------|-----------------|
| Feature contract | `FEATURE_COLS`, `CONTINUOUS_LABEL_COLS`, horizons |
| Derivatives | Funding/OI z-scores |
| Feature engineering | `add_features` |
| Labels / targets | `compute_market_labels` |
| Inference helpers | `feature_config.json` builders |
| Delta data | `fetch_candles`, `fetch_history_bundle` |
| Training pipeline | `MarketTransformer`, train loop, ONNX export |
| Training runner | `run_training`, `run_all_training` |
| Configure & train | `resolutions`, `epochs`, `history_days` |

Permanent edits: change repo `.py` files under `feature_store/transformer_btcusd/` and
`scripts/colab/`, then regenerate::

    python scripts/colab/build_standalone_notebook.py

**Colab setup:** Runtime → Change runtime type → **T4 GPU** (recommended).

Each TF exports to its own subdirectory under `export_dir`:

```
export/
├── JackSparrow_Transformer_BTCUSD_5m/
│   ├── metadata_transformer.json
│   ├── btcusd_5m_transformer.onnx
│   └── feature_config.json
└── ...
```

Copy each subdirectory into `agent/model_storage/` after training.

## Setup

In [ ]:
# Colab ships torch/pandas/numpy; only install what training needs beyond that.
!pip install -q --upgrade-strategy only-if-needed pyarrow onnx onnxruntime requests


In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected — training will run on CPU and be much slower.")
    print("Runtime -> Change runtime type -> select a GPU (T4), then re-run this cell.")


## Feature contract (agent integration)

In [ ]:
"""Constants shared between per-TF Colab training and agent inference."""

from __future__ import annotations

from typing import Any, Dict, Tuple

# Bump when FEATURE_COLS or semantics change (requires retrain + re-export).
FEATURE_CONTRACT_VERSION = "transformer_btcusd_per_tf_features_v1"

SUPPORTED_RESOLUTIONS: Tuple[str, ...] = ("5m", "15m", "30m", "1h", "2h")

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
}

TF_KEYS: Tuple[str, ...] = tuple(f"tf_{r}" for r in SUPPORTED_RESOLUTIONS)

FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "rv_16",
    "rv_96",
    "ema50_dist_pct",
    "macd_hist",
    "rsi_14",
    "adx_14",
    "obv_z",
    "vol_z",
    "body_ratio",
    "upper_wick_ratio",
    "lower_wick_ratio",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "dist_to_resistance_pct",
    "dist_to_support_pct",
    "funding_rate",
    "oi_z",
    "funding_zscore",
    "funding_mom",
    "funding_rate_roc",
    "oi_change_2",
    "oi_delta_z",
    "oi_price_divergence",
    "oi_acceleration",
    "funding_x_oi",
)

RETURN_COL = "future_return"

PATH_LABEL_COLS: Tuple[str, ...] = (
    "mfe",
    "mae",
    "future_volatility",
    "trend_strength",
    "drawdown_before_mfe",
    "future_oi_change_pct",
    "future_volume_change_pct",
)

CONTINUOUS_LABEL_COLS: Tuple[str, ...] = (RETURN_COL,) + PATH_LABEL_COLS

PATH_LABEL_HORIZON_BARS: int = 8

# Reference Colab notebook (btcusd_15m_transformer) uses 32 bars on 15m (~8h wall-clock).
REFERENCE_LABEL_HORIZON_MINUTES: int = 480

REGIME_NAMES: Dict[int, str] = {
    0: "LOW",
    1: "NORMAL",
    2: "HIGH",
    3: "EXTREME",
}

# Minimum test-set correlation for future_return before ONNX export.
MIN_EXPORT_RETURN_CORR: Dict[str, float] = {
    "5m": 0.03,
    "15m": 0.04,
    "30m": 0.03,
    "1h": 0.02,
    "2h": 0.02,
}

# Stricter optional targets for promotion-ready bundles (warn-only unless agent flag set).
PROMOTION_RETURN_CORR: Dict[str, float] = {
    "5m": 0.05,
    "15m": 0.06,
    "30m": 0.05,
    "1h": 0.04,
    "2h": 0.04,
}
PROMOTION_VOL_CORR: float = 0.15
PROMOTION_REGIME_ACCURACY: float = 0.35

EXPORT_QUALITY_DISCLAIMER = (
    "Sanity gates detect broken exports, not trading edge. "
    "Promotion tier targets are informational."
)

TRANSFORMER_METADATA_FILENAME = "metadata_transformer.json"
TRANSFORMER_FEATURE_CONFIG_FILENAME = "feature_config.json"

def model_family_for_resolution(resolution: str) -> str:
    """Canonical model_family string for a TF bundle."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"jacksparrow_transformer_btcusd_{res}"

def onnx_filename_for_resolution(resolution: str) -> str:
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"btcusd_{res}_transformer.onnx"

def bundle_dir_name(resolution: str) -> str:
    res = resolution.strip().lower()
    return f"JackSparrow_Transformer_BTCUSD_{res}"

def label_horizon_bars_for_resolution(resolution_minutes: int) -> int:
    """Forward label window in bars (~8h wall-clock; 32 bars on 15m per reference notebook)."""
    return max(1, int(round(REFERENCE_LABEL_HORIZON_MINUTES / resolution_minutes)))

def default_training_config(resolution: str) -> Dict[str, Any]:
    """Default Colab training config for a single TF model."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    label_horizon = label_horizon_bars_for_resolution(minutes)
    return {
        "symbol": "BTCUSD",
        "resolution": res,
        "resolution_minutes": minutes,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "atr_period": 14,
        "return_horizon_bars": label_horizon,
        "path_label_horizon_bars": label_horizon,
        "mae_floor_atr_mult": 0.25,
        "vol_regime_quantiles": [0.25, 0.5, 0.75],
        "window_len": 128,
        "stride": 8,
        "train_frac": 0.65,
        "val_frac": 0.15,
        "embargo_bars": label_horizon,
        "batch_size": 128,
        "epochs": 120,
        "lr": 1e-4,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dropout": 0.25,
        "weight_decay": 1e-2,
        "early_stop_patience": 12,
        "early_stopping_enabled": True,
        "min_derivatives_coverage": 0.5,
        "derivatives_coverage_warn": 0.9,
        "min_export_return_corr": MIN_EXPORT_RETURN_CORR.get(res, 0.02),
        "default_threshold": 0.005,
        "seed": 42,
    }

def max_label_horizon_bars(
    return_horizon_bars: int = 1,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
) -> int:
    """Maximum forward bars across all training labels."""
    return max(int(return_horizon_bars), int(path_label_horizon_bars))

def scale_period(period: int, resolution_minutes: int, *, base_minutes: int = 5) -> int:
    """Scale indicator lookback to preserve wall-clock semantics across TFs."""
    return max(1, int(round(period * resolution_minutes / base_minutes)))


## Derivatives features

In [ ]:
"""Funding and OI derivative features scaled per resolution."""

from __future__ import annotations

import numpy as np
import pandas as pd

_EPS = 1e-9

def compute_funding_derivatives(
    fund_rate: pd.Series,
    ret_2: pd.Series,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """Funding z-score, momentum, and rate-of-change on native TF bars."""
    z_window = scale_period(16, resolution_minutes)
    roc_diff = max(1, scale_period(1, resolution_minutes))
    fz_mean = fund_rate.rolling(z_window, min_periods=5).mean()
    fz_std = fund_rate.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_zscore = ((fund_rate - fz_mean) / fz_std).fillna(0.0).clip(-4.0, 4.0)
    funding_mom = funding_zscore * ret_2
    diff = fund_rate.diff(roc_diff)
    roc_mu = diff.rolling(z_window, min_periods=5).mean()
    roc_std = diff.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_rate_roc = ((diff - roc_mu) / roc_std).fillna(0.0).clip(-4.0, 4.0)
    return pd.DataFrame(
        {
            "funding_zscore": funding_zscore,
            "funding_mom": funding_mom.fillna(0.0),
            "funding_rate_roc": funding_rate_roc,
        }
    )

def compute_oi_derivatives(
    primary: pd.DataFrame,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """OI-derived features on native TF bars."""
    n = len(primary)
    zero = pd.DataFrame(
        {
            "oi_change_2": np.zeros(n),
            "oi_delta_z": np.zeros(n),
            "oi_price_divergence": np.zeros(n),
            "oi_acceleration": np.zeros(n),
            "oi_zscore": np.zeros(n),
        },
        index=primary.index,
    )
    if "open_interest" not in primary.columns:
        return zero

    z_window = scale_period(16, resolution_minutes)
    change_window = max(1, scale_period(2, resolution_minutes))

    oi_s = primary["open_interest"].astype(float)
    if oi_s.notna().sum() == 0 or float(oi_s.max()) < _EPS:
        return zero

    oi_mu = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_std = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_zscore = ((oi_s - oi_mu) / oi_std).fillna(0.0).clip(-4.0, 4.0)

    oi_lagged = oi_s.shift(change_window).bfill().fillna(oi_s)
    oi_change_2 = (
        ((oi_s - oi_lagged) / (oi_lagged.abs() + _EPS)).fillna(0.0).clip(-0.05, 0.05)
    )

    close_s = primary["close"].astype(float)
    close_lagged = close_s.shift(change_window).bfill().fillna(close_s)
    ret_2 = ((close_s - close_lagged) / (close_lagged.abs() + _EPS)).fillna(0.0)
    oi_price_divergence = (
        np.sign(oi_change_2.values) * -np.sign(ret_2.values)
    ).astype(np.float32)
    oi_acceleration = oi_change_2.diff().fillna(0.0).clip(-0.02, 0.02)

    oi_delta_1 = oi_s.diff(1).fillna(0.0)
    oi_delta_mu = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_delta_std = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_delta_z = ((oi_delta_1 - oi_delta_mu) / oi_delta_std).fillna(0.0).clip(-4.0, 4.0)

    return pd.DataFrame(
        {
            "oi_zscore": oi_zscore,
            "oi_change_2": oi_change_2,
            "oi_price_divergence": pd.Series(oi_price_divergence).fillna(0.0),
            "oi_acceleration": oi_acceleration,
            "oi_delta_z": oi_delta_z,
        },
        index=primary.index,
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0)


## Feature engineering

In [ ]:
"""Causal per-TF feature engineering for BTCUSD transformers."""

from __future__ import annotations

from typing import Optional

import numpy as np
import pandas as pd

def assemble_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Merge OHLCV with funding/OI using the same path as live inference."""
    out = df.copy()
    if "timestamp" in out.columns and "time" not in out.columns:
        out["time"] = pd.to_datetime(out["timestamp"], utc=True)
    elif "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    else:
        raise ValueError("OHLCV frame requires timestamp or time column")

    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"OHLCV frame missing columns: {missing}")

    out = out.sort_values("time").reset_index(drop=True)

    if funding_df is not None and not funding_df.empty:
        fund = funding_df.copy()
        if "timestamp" in fund.columns:
            fund["time"] = pd.to_datetime(fund["timestamp"], utc=True)
        rate_col = "funding_rate" if "funding_rate" in fund.columns else "close"
        fund = fund[["time", rate_col]].rename(columns={rate_col: "funding_rate"})
        out = pd.merge_asof(
            out.sort_values("time"),
            fund.sort_values("time"),
            on="time",
            direction="backward",
        )
    elif "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if oi_df is not None and not oi_df.empty:
        oi = oi_df.copy()
        if "timestamp" in oi.columns:
            oi["time"] = pd.to_datetime(oi["timestamp"], utc=True)
        oi_col = None
        for candidate in ("open_interest", "oi_contracts", "close"):
            if candidate in oi.columns:
                oi_col = candidate
                break
        if oi_col is not None:
            oi = oi[["time", oi_col]].rename(columns={oi_col: "open_interest"})
            out = pd.merge_asof(
                out.sort_values("time"),
                oi.sort_values("time"),
                on="time",
                direction="backward",
            )
    elif "open_interest" not in out.columns:
        out["open_interest"] = np.nan

    if "funding_rate" in out.columns:
        out["funding_rate"] = out["funding_rate"].ffill().bfill()
    if "open_interest" in out.columns:
        out["open_interest"] = out["open_interest"].ffill().bfill()

    return out

def prepare_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Alias for assemble_raw_frame (agent inference entry point)."""
    return assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)

def add_features(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    rsi_period: int = 14,
    adx_period: int = 14,
    ema_period: int = 50,
    macd_fast: int = 12,
    macd_slow: int = 26,
    macd_signal: int = 9,
) -> pd.DataFrame:
    """Compute causal features on a native TF grid."""
    out = df.copy()
    rv_short = scale_period(16, resolution_minutes)
    rv_long = scale_period(96, resolution_minutes)
    sr_window = scale_period(96, resolution_minutes)
    obv_window = scale_period(96, resolution_minutes)
    vol_window = scale_period(96, resolution_minutes)
    ret2_bars = max(1, scale_period(2, resolution_minutes))

    atr_period = scale_period(atr_period, resolution_minutes)
    rsi_period = scale_period(rsi_period, resolution_minutes)
    adx_period = scale_period(adx_period, resolution_minutes)
    ema_period = scale_period(ema_period, resolution_minutes)
    macd_fast = scale_period(macd_fast, resolution_minutes)
    macd_slow = scale_period(macd_slow, resolution_minutes)
    macd_signal = scale_period(macd_signal, resolution_minutes)

    out["ret_1"] = np.log(out["close"] / out["close"].shift(1))

    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(atr_period).mean()

    out["rv_16"] = out["ret_1"].rolling(rv_short).std()
    out["rv_96"] = out["ret_1"].rolling(rv_long).std()

    out["ema50"] = out["close"].ewm(span=ema_period, adjust=False).mean()
    out["ema50_dist_pct"] = (out["close"] - out["ema50"]) / out["ema50"]

    ema_fast_s = out["close"].ewm(span=macd_fast, adjust=False).mean()
    ema_slow_s = out["close"].ewm(span=macd_slow, adjust=False).mean()
    macd_line = ema_fast_s - ema_slow_s
    macd_signal_line = macd_line.ewm(span=macd_signal, adjust=False).mean()
    out["macd_hist"] = macd_line - macd_signal_line

    delta = out["close"].diff()
    gain = delta.clip(lower=0).rolling(rsi_period).mean()
    loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()
    rs = gain / (loss + 1e-9)
    out["rsi_14"] = 100 - (100 / (1 + rs))

    up_move = out["high"].diff()
    down_move = -out["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr_for_di = tr.rolling(adx_period).mean()
    plus_di = 100 * pd.Series(plus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    minus_di = 100 * pd.Series(minus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    out["adx_14"] = dx.rolling(adx_period).mean()

    obv_raw = (np.sign(out["close"].diff()) * out["volume"]).fillna(0).cumsum()
    out["obv_z"] = (obv_raw - obv_raw.rolling(obv_window).mean()) / (
        obv_raw.rolling(obv_window).std() + 1e-9
    )

    out["vol_z"] = (out["volume"] - out["volume"].rolling(vol_window).mean()) / (
        out["volume"].rolling(vol_window).std() + 1e-9
    )

    rng = (out["high"] - out["low"]).replace(0, np.nan)
    out["body_ratio"] = (out["close"] - out["open"]) / rng
    out["upper_wick_ratio"] = (
        out["high"] - out[["open", "close"]].max(axis=1)
    ) / rng
    out["lower_wick_ratio"] = (
        out[["open", "close"]].min(axis=1) - out["low"]
    ) / rng

    out["hour"] = out["time"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow"] = out["time"].dt.dayofweek
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)

    rolling_high = out["high"].rolling(sr_window).max()
    rolling_low = out["low"].rolling(sr_window).min()
    out["dist_to_resistance_pct"] = (rolling_high - out["close"]) / out["close"]
    out["dist_to_support_pct"] = (out["close"] - rolling_low) / out["close"]

    if "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if "open_interest" in out.columns:
        oi_z_window = scale_period(96, resolution_minutes)
        out["oi_z"] = (out["open_interest"] - out["open_interest"].rolling(oi_z_window).mean()) / (
            out["open_interest"].rolling(oi_z_window).std() + 1e-9
        )
    else:
        out["oi_z"] = np.nan

    ret_2 = out["close"].pct_change(ret2_bars)
    fund_rate = out["funding_rate"].fillna(0.0)
    funding_deriv = compute_funding_derivatives(
        fund_rate, ret_2, resolution_minutes=resolution_minutes
    )
    for col in funding_deriv.columns:
        out[col] = funding_deriv[col].values

    oi_deriv = compute_oi_derivatives(out, resolution_minutes=resolution_minutes)
    out["oi_change_2"] = oi_deriv["oi_change_2"]
    out["oi_delta_z"] = oi_deriv["oi_delta_z"]
    out["oi_price_divergence"] = oi_deriv["oi_price_divergence"]
    out["oi_acceleration"] = oi_deriv["oi_acceleration"]
    out["funding_x_oi"] = out["funding_zscore"] * oi_deriv["oi_zscore"]

    return out

def build_feature_matrix(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    dropna: bool = True,
) -> pd.DataFrame:
    """Return feature columns ready for windowing."""
    feat = add_features(df, resolution_minutes=resolution_minutes, atr_period=atr_period)
    if dropna:
        feat = feat.dropna().reset_index(drop=True)
    return feat

def latest_closed_feature_row(feat_df: pd.DataFrame) -> pd.Series:
    """Closed-bar row used for diagnostics (second-to-last after dropna)."""
    if len(feat_df) < 2:
        raise ValueError("Need at least 2 feature rows for closed-bar semantics")
    return feat_df.iloc[-2]

def validate_feature_columns(
    feat_df: pd.DataFrame,
    *,
    require_finite_closed_bar: bool = False,
) -> None:
    missing = [c for c in FEATURE_COLS if c not in feat_df.columns]
    if missing:
        raise ValueError(f"Feature matrix missing columns: {missing}")
    if require_finite_closed_bar and len(feat_df) >= 2:
        closed = latest_closed_feature_row(feat_df)
        for col in FEATURE_COLS:
            val = closed[col]
            if not np.isfinite(float(val)):
                raise ValueError(f"Non-finite closed-bar value for {col}: {val}")


## Labels / targets

In [ ]:
"""Forward-looking market labels for per-TF transformer training."""

from __future__ import annotations

from typing import Dict

import numpy as np
import pandas as pd

def compute_market_labels(
    df: pd.DataFrame,
    *,
    return_horizon_bars: int = 1,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
    mae_floor_atr_mult: float,
) -> pd.DataFrame:
    """Compute single-bar return target and path/risk labels on native TF grid."""
    path_horizon = int(path_label_horizon_bars)
    ret_horizon = int(return_horizon_bars)
    max_horizon = max_label_horizon_bars(ret_horizon, path_horizon)

    out_df = df.copy()
    n = len(out_df)
    close = out_df["close"].values
    high = out_df["high"].values
    low = out_df["low"].values
    atr = out_df["atr"].values
    volume = out_df["volume"].values
    has_oi = "open_interest" in out_df.columns
    oi = out_df["open_interest"].values if has_oi else np.full(n, np.nan)

    path_cols = [
        "mfe",
        "mae",
        "future_volatility",
        "trend_strength",
        "drawdown_before_mfe",
        "future_oi_change_pct",
        "future_volume_change_pct",
    ]
    out: dict[str, np.ndarray] = {
        RETURN_COL: np.full(n, np.nan),
        **{c: np.full(n, np.nan) for c in path_cols},
    }

    for i in range(n - max_horizon):
        entry = close[i]

        if ret_horizon > 0 and i + ret_horizon < n:
            out[RETURN_COL][i] = (close[i + ret_horizon] - entry) / entry

        fwd_close = close[i + 1 : i + path_horizon + 1]
        fwd_high = high[i + 1 : i + path_horizon + 1]
        fwd_low = low[i + 1 : i + path_horizon + 1]

        fwd_rets = np.log(fwd_close / np.concatenate(([entry], fwd_close[:-1])))
        out["future_volatility"][i] = fwd_rets.std()

        favorable = (fwd_high - entry) / entry
        adverse = (entry - fwd_low) / entry
        mfe_idx = int(np.argmax(favorable))
        out["mfe"][i] = favorable[mfe_idx]
        out["mae"][i] = adverse[int(np.argmax(adverse))]

        if out["mfe"][i] >= mae_floor_atr_mult * atr[i] / entry:
            out["drawdown_before_mfe"][i] = (
                adverse[: mfe_idx + 1].max() if mfe_idx > 0 else 0.0
            )

        out["trend_strength"][i] = abs(fwd_close[-1] - entry) / (atr[i] + 1e-9)

        if has_oi and not np.isnan(oi[i]) and oi[i] != 0:
            out["future_oi_change_pct"][i] = (oi[i + path_horizon] - oi[i]) / (
                abs(oi[i]) + 1e-9
            )
        past_start = max(0, i - path_horizon)
        past_vol_mean = volume[past_start : i + 1].mean()
        out["future_volume_change_pct"][i] = (
            volume[i + 1 : i + path_horizon + 1].mean() - past_vol_mean
        ) / (past_vol_mean + 1e-9)

    for col, values in out.items():
        out_df[col] = values
    return out_df

def trim_label_tail(
    df: pd.DataFrame,
    *,
    return_horizon_bars: int = 1,
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
) -> pd.DataFrame:
    """Drop rows without complete forward labels."""
    max_horizon = max_label_horizon_bars(return_horizon_bars, path_label_horizon_bars)
    return df.iloc[: -(max_horizon + 1)].reset_index(drop=True)

def active_training_label_cols() -> tuple[str, ...]:
    return CONTINUOUS_LABEL_COLS

def label_nan_summary(df: pd.DataFrame) -> Dict[str, float]:
    """Per-label NaN rates for notebook diagnostics."""
    cols = [c for c in CONTINUOUS_LABEL_COLS if c in df.columns]
    if not cols:
        return {}
    return df[cols].isna().mean().round(4).to_dict()


## Inference and export helpers

In [ ]:
"""Window building and label un-standardization for per-TF transformer inference."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Mapping, Sequence, Tuple

import numpy as np

def load_feature_config(path: Path) -> Dict[str, Any]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise ValueError(f"{path} must contain a JSON object")
    return raw

def resolve_feature_config(bundle_dir: Path) -> Dict[str, Any]:
    cfg_path = bundle_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME
    if not cfg_path.is_file():
        raise FileNotFoundError(
            f"Missing {TRANSFORMER_FEATURE_CONFIG_FILENAME} in {bundle_dir}"
        )
    return load_feature_config(cfg_path)

def zscore_window(window: np.ndarray) -> np.ndarray:
    """Per-window z-score (matches Colab WindowDataset)."""
    mu = window.mean(axis=0, keepdims=True)
    sd = window.std(axis=0, keepdims=True) + 1e-6
    return ((window - mu) / sd).astype(np.float32)

def build_inference_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a single (1, window_len, n_features) tensor from feature matrix values."""
    if feat_values.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} feature rows, got {feat_values.shape[0]}"
        )
    window = feat_values[-window_len:, :].astype(np.float32)
    if not np.isfinite(window).all():
        raise ValueError("Feature window contains non-finite values")
    normed = zscore_window(window)
    return normed[np.newaxis, :, :]

def unstandardize_continuous(
    pred_z: np.ndarray,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    *,
    label_cols: Sequence[str] = CONTINUOUS_LABEL_COLS,
) -> Dict[str, float]:
    """Map standardized ONNX output back to real units."""
    mean = np.asarray(label_mean, dtype=np.float64)
    std = np.asarray(label_std, dtype=np.float64)
    flat = np.asarray(pred_z, dtype=np.float64).reshape(-1)
    if flat.shape[0] != len(label_cols):
        raise ValueError(
            f"Expected {len(label_cols)} continuous outputs, got {flat.shape[0]}"
        )
    real = flat * std + mean
    return {str(col): float(val) for col, val in zip(label_cols, real)}

def softmax(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64).reshape(-1)
    x = x - x.max()
    exp = np.exp(x)
    return exp / (exp.sum() + 1e-12)

def parse_regime_prediction(
    regime_logits: np.ndarray,
    regime_names: Mapping[str, str],
) -> Tuple[int, str, Dict[str, float]]:
    probs = softmax(regime_logits)
    idx = int(np.argmax(probs))
    name = regime_names.get(str(idx), regime_names.get(idx, f"CLASS_{idx}"))
    prob_map = {
        regime_names.get(str(i), f"CLASS_{i}"): float(probs[i])
        for i in range(len(probs))
    }
    return idx, str(name), prob_map

def feature_config_from_training_export(
    *,
    feature_cols: Sequence[str],
    window_len: int,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    q_edges: Sequence[float],
    config: Mapping[str, Any],
) -> Dict[str, Any]:
    """Build feature_config.json payload written by Colab export cell."""
    return {
        "feature_contract_version": FEATURE_CONTRACT_VERSION,
        "feature_cols": list(feature_cols),
        "window_len": int(window_len),
        "continuous_label_cols": list(CONTINUOUS_LABEL_COLS),
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "regime_names": {"0": "LOW", "1": "NORMAL", "2": "HIGH", "3": "EXTREME"},
        "vol_regime_quantile_edges": [float(x) for x in q_edges],
        "config": dict(config),
    }

def metadata_from_training_export(
    *,
    resolution: str,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    config: Mapping[str, Any],
    test_metrics: Mapping[str, Any] | None = None,
    export_quality: Mapping[str, Any] | None = None,
) -> Dict[str, Any]:
    """Build metadata_transformer.json for a per-TF bundle."""
    res = resolution.strip().lower()
    cfg = dict(config)
    meta: Dict[str, Any] = {
        "version": "transformer_per_tf_v1",
        "model_name": f"jacksparrow_transformer_BTCUSD_{res}",
        "model_family": model_family_for_resolution(res),
        "symbol": str(cfg.get("symbol") or "BTCUSD"),
        "resolution": res,
        "resolution_minutes": int(cfg.get("resolution_minutes") or 15),
        "onnx_filename": onnx_filename_for_resolution(res),
        "feature_config_filename": TRANSFORMER_FEATURE_CONFIG_FILENAME,
        "return_horizon_bars": int(cfg.get("return_horizon_bars") or 1),
        "path_label_horizon_bars": int(cfg.get("path_label_horizon_bars") or 8),
        "atr_period": int(cfg.get("atr_period") or 14),
        "default_threshold": float(cfg.get("default_threshold") or 0.005),
        "primary_signal_mode": "returns",
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "test_metrics": dict(test_metrics or {}),
        "training_config": cfg,
    }
    if export_quality:
        meta["export_quality"] = dict(export_quality)
    return meta

def training_config_for_resolution(resolution: str) -> Dict[str, Any]:
    return default_training_config(resolution)


## Delta Exchange India data

In [ ]:
"""Public Delta India history fetch for Colab transformer training."""

from __future__ import annotations

import time
from datetime import datetime, timedelta, timezone
from typing import Any, Dict

import pandas as pd
import requests

_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "1d": 1440,
}

def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
) -> pd.DataFrame:
    """Paginated pull of OHLC-style candles from Delta India public history API."""
    url = f"{base_url}/v2/history/candles"
    all_rows = []
    resolution_minutes = _RESOLUTION_MINUTES[resolution]
    chunk_seconds = resolution_minutes * 60 * 2000

    cur_start = start_ts
    while cur_start < end_ts:
        cur_end = min(cur_start + chunk_seconds, end_ts)
        params = {
            "symbol": symbol,
            "resolution": resolution,
            "start": cur_start,
            "end": cur_end,
        }
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        rows = payload.get("result", [])
        if rows:
            all_rows.extend(rows)
        cur_start = cur_end
        time.sleep(0.2)

    if not all_rows:
        raise RuntimeError(
            f"No candle data returned for symbol={symbol}; check symbol/resolution/date range."
        )

    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)
    return df

def validate_derivatives_coverage(
    df: pd.DataFrame,
    *,
    min_coverage: float = 0.5,
    warn_coverage: float = 0.9,
) -> Dict[str, Any]:
    """Check funding/OI non-null coverage over the OHLCV timeline."""
    n = len(df)
    if n == 0:
        raise ValueError("Cannot validate derivatives coverage on empty frame")

    report: Dict[str, Any] = {"rows": n, "columns": {}}
    for col in ("funding_rate", "open_interest"):
        if col not in df.columns:
            report["columns"][col] = {"coverage": 0.0, "status": "missing"}
            continue
        coverage = float(df[col].notna().mean())
        status = "ok"
        if coverage < min_coverage:
            status = "error"
        elif coverage < warn_coverage:
            status = "warn"
        report["columns"][col] = {
            "coverage": round(coverage, 4),
            "status": status,
        }

    worst = min(v["coverage"] for v in report["columns"].values())
    report["worst_coverage"] = worst
    if worst < min_coverage:
        raise ValueError(
            "Derivatives coverage below minimum "
            f"({worst:.1%} < {min_coverage:.0%}): {report['columns']}"
        )
    return report

def fetch_history_bundle(
    *,
    symbol: str,
    resolution: str,
    history_days: int,
    base_url: str,
    min_derivatives_coverage: float = 0.5,
    derivatives_coverage_warn: float = 0.9,
) -> pd.DataFrame:
    """Fetch OHLCV + funding + OI merged frame (notebook-compatible)."""
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=history_days)
    start_ts, end_ts = int(start_dt.timestamp()), int(end_dt.timestamp())

    raw_df = fetch_candles(symbol, resolution, start_ts, end_ts, base_url)

    try:
        funding_df = fetch_candles(
            f"FUNDING:{symbol}", resolution, start_ts, end_ts, base_url
        )
        funding_df = funding_df[["time", "close"]].rename(columns={"close": "funding_rate"})
    except Exception:
        funding_df = pd.DataFrame({"time": [], "funding_rate": []})

    try:
        oi_df = fetch_candles(f"OI:{symbol}", resolution, start_ts, end_ts, base_url)
        oi_df = oi_df[["time", "close"]].rename(columns={"close": "open_interest"})
    except Exception:
        oi_df = pd.DataFrame({"time": [], "open_interest": []})

    raw_df = assemble_raw_frame(raw_df, funding_df=funding_df, oi_df=oi_df)

    report = validate_derivatives_coverage(
        raw_df,
        min_coverage=min_derivatives_coverage,
        warn_coverage=derivatives_coverage_warn,
    )
    for col, info in report["columns"].items():
        if info["status"] == "warn":
            print(
                f"WARNING: {col} coverage {info['coverage']:.1%} "
                f"below recommended {derivatives_coverage_warn:.0%}"
            )
        elif info["status"] == "ok":
            print(f"{col} coverage: {info['coverage']:.1%}")

    return raw_df


## Training pipeline

In [ ]:
"""Training loop and model definitions for per-TF BTCUSD transformer Colab workflow."""

from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

def set_training_seed(seed: int) -> None:
    """Set RNG seeds for reproducible Colab runs."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_windows(
    df: pd.DataFrame,
    feature_cols: Sequence[str],
    label_cols: Sequence[str],
    window_len: int,
    stride: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Stack overlapping feature windows and end-of-window labels."""
    x_list: List[np.ndarray] = []
    y_list: List[np.ndarray] = []
    values = df[list(feature_cols)].values.astype(np.float32)
    labels = df[list(label_cols)].values.astype(np.float64)
    for end in range(window_len, len(df), stride):
        start = end - window_len
        x_list.append(values[start:end])
        y_list.append(labels[end - 1])
    return np.array(x_list), np.array(y_list)

def split_purged_windows(
    x_all: np.ndarray,
    y_all: np.ndarray,
    *,
    train_frac: float,
    val_frac: float,
    embargo_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered train/val/test split with embargo gaps between segments."""
    n = len(x_all)
    train_end = int(n * train_frac)
    val_end = train_end + int(n * val_frac)
    embargo = embargo_bars

    splits = {
        "x_train": x_all[:train_end],
        "y_train": y_all[:train_end],
        "x_val": x_all[train_end + embargo : val_end],
        "y_val": y_all[train_end + embargo : val_end],
        "x_test": x_all[val_end + embargo : n],
        "y_test": y_all[val_end + embargo : n],
    }
    if len(splits["x_train"]) == 0 or len(splits["x_val"]) == 0:
        raise ValueError(
            f"Insufficient windows after split: train={len(splits['x_train'])}, "
            f"val={len(splits['x_val'])}, test={len(splits['x_test'])}"
        )
    return splits

def fit_label_stats(y_train: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Train-split mean/std for label standardization (NaNs excluded from fit)."""
    mean = np.nanmean(y_train, axis=0)
    std = np.nanstd(y_train, axis=0) + 1e-9
    return mean, std

def standardize_labels(
    y: np.ndarray,
    label_mean: np.ndarray,
    label_std: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Z-score labels and build per-target validity masks."""
    mask = (~np.isnan(y)).astype(np.float32)
    yz = np.where(np.isnan(y), 0.0, (y - label_mean) / label_std).astype(np.float32)
    return yz, mask

def fit_vol_regime_edges(
    y_train: np.ndarray,
    quantiles: Sequence[float],
) -> np.ndarray:
    """Quantile cutoffs for future_volatility regime classification."""
    fv_idx = CONTINUOUS_LABEL_COLS.index("future_volatility")
    return np.nanquantile(y_train[:, fv_idx], quantiles)

def to_vol_regime(y: np.ndarray, q_edges: np.ndarray) -> np.ndarray:
    """Map raw labels to 4-class volatility regime indices."""
    fv_idx = CONTINUOUS_LABEL_COLS.index("future_volatility")
    return np.digitize(y[:, fv_idx], q_edges).astype(np.int64)

class WindowDataset(Dataset):
    """Per-window z-scored features with standardized multi-task targets."""

    def __init__(
        self,
        x: np.ndarray,
        y_z: np.ndarray,
        mask: np.ndarray,
        regime: np.ndarray,
    ) -> None:
        self.x = x
        self.y_z = y_z
        self.mask = mask
        self.regime = regime

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(
        self, i: int
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        w = zscore_window(self.x[i])
        return (
            torch.tensor(w, dtype=torch.float32),
            torch.tensor(self.y_z[i], dtype=torch.float32),
            torch.tensor(self.mask[i], dtype=torch.float32),
            torch.tensor(self.regime[i], dtype=torch.long),
        )

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512) -> None:
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]

class MarketTransformer(nn.Module):
    """Multi-task market-understanding encoder (continuous + vol-regime heads)."""

    def __init__(
        self,
        n_features: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dropout: float,
        max_len: int,
        n_continuous: int,
        n_regime_classes: int = 4,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        shared_dim = d_model // 2
        self.shared = nn.Sequential(
            nn.Linear(d_model, shared_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.continuous_head = nn.Linear(shared_dim, n_continuous)
        self.regime_head = nn.Linear(shared_dim, n_regime_classes)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.input_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)
        h = self.norm(h)
        pooled = h[:, -1, :]
        shared = self.shared(pooled)
        return self.continuous_head(shared), self.regime_head(shared)

def compute_multitask_loss(
    continuous_pred: torch.Tensor,
    regime_logits: torch.Tensor,
    yb_z: torch.Tensor,
    mb: torch.Tensor,
    rb: torch.Tensor,
    log_vars: nn.Parameter,
    *,
    n_continuous: int,
) -> torch.Tensor:
    """Uncertainty-weighted multi-task loss over continuous heads + regime CE."""
    losses: List[torch.Tensor] = []
    for j in range(n_continuous):
        diff = (continuous_pred[:, j] - yb_z[:, j]) ** 2
        masked = (diff * mb[:, j]).sum() / (mb[:, j].sum() + 1e-6)
        precision = torch.exp(-log_vars[j])
        losses.append(precision * masked + log_vars[j])
    ce = nn.functional.cross_entropy(regime_logits, rb)
    precision = torch.exp(-log_vars[n_continuous])
    losses.append(precision * ce + log_vars[n_continuous])
    return sum(losses)

@dataclass
class TrainingResult:
    """Best checkpoint and training metadata."""

    best_val_loss: float
    best_epoch: int
    model_state: Dict[str, torch.Tensor]
    log_vars: torch.Tensor
    history: List[Tuple[int, float, float]] = field(default_factory=list)
    stopped_at_epoch: int = 0

def train_transformer(
    model: MarketTransformer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: Dict[str, Any],
    *,
    device: torch.device,
    n_continuous: int | None = None,
) -> TrainingResult:
    """Train with AdamW, optional LR scheduler, and early stopping."""
    n_cont = n_continuous or len(CONTINUOUS_LABEL_COLS)
    log_vars = nn.Parameter(torch.zeros(n_cont + 1, device=device))
    params = list(model.parameters()) + [log_vars]
    optimizer = torch.optim.AdamW(
        params,
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    best_val = float("inf")
    best_epoch = 0
    best_state: Optional[Dict[str, torch.Tensor]] = None
    best_log_vars: Optional[torch.Tensor] = None
    epochs_since_improvement = 0
    history: List[Tuple[int, float, float]] = []

    for epoch in range(config["epochs"]):
        model.train()
        train_loss = 0.0
        for xb, yb_z, mb, rb in train_loader:
            xb, yb_z, mb, rb = (
                xb.to(device),
                yb_z.to(device),
                mb.to(device),
                rb.to(device),
            )
            optimizer.zero_grad()
            cont, reg = model(xb)
            loss = compute_multitask_loss(
                cont, reg, yb_z, mb, rb, log_vars, n_continuous=n_cont
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
            train_loss += float(loss.item())

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb_z, mb, rb in val_loader:
                xb, yb_z, mb, rb = (
                    xb.to(device),
                    yb_z.to(device),
                    mb.to(device),
                    rb.to(device),
                )
                cont, reg = model(xb)
                val_loss += float(
                    compute_multitask_loss(
                        cont, reg, yb_z, mb, rb, log_vars, n_continuous=n_cont
                    ).item()
                )
        val_loss /= max(len(val_loader), 1)
        train_loss /= max(len(train_loader), 1)
        history.append((epoch + 1, train_loss, val_loss))

        prev_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        lr_note = f" lr->{new_lr:.2e}" if new_lr < prev_lr else ""

        marker = ""
        if val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch + 1
            epochs_since_improvement = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best_log_vars = log_vars.detach().cpu()
            marker = " *best*"
        else:
            epochs_since_improvement += 1

        print(
            f"epoch {epoch + 1:03d}  train={train_loss:.4f}  val={val_loss:.4f}{marker}{lr_note}"
        )

        if (
            config.get("early_stopping_enabled", False)
            and epochs_since_improvement >= config.get("early_stop_patience", 10)
        ):
            print(
                f"Early stopping: no val improvement in "
                f"{config['early_stop_patience']} epochs."
            )
            break

    if best_state is None or best_log_vars is None:
        raise RuntimeError("Training did not produce a checkpoint")

    stopped_at = history[-1][0] if history else best_epoch
    print(f"Training finished at epoch {stopped_at}, best epoch {best_epoch}")

    return TrainingResult(
        best_val_loss=best_val,
        best_epoch=best_epoch,
        model_state=best_state,
        log_vars=best_log_vars,
        history=history,
        stopped_at_epoch=stopped_at,
    )

@dataclass
class TargetMetrics:
    name: str
    mae: float
    corr: float
    n: int

def evaluate_continuous_targets(
    model: MarketTransformer,
    loader: DataLoader,
    *,
    device: torch.device,
    label_mean: np.ndarray,
    label_std: np.ndarray,
    label_cols: Sequence[str] = CONTINUOUS_LABEL_COLS,
) -> List[TargetMetrics]:
    """Per-target MAE and correlation on un-standardized predictions."""
    model.eval()
    pred_z_list: List[np.ndarray] = []
    true_z_list: List[np.ndarray] = []
    mask_list: List[np.ndarray] = []

    with torch.no_grad():
        for xb, yb_z, mb, _rb in loader:
            xb = xb.to(device)
            cont, _ = model(xb)
            pred_z_list.append(cont.cpu().numpy())
            true_z_list.append(yb_z.numpy())
            mask_list.append(mb.numpy())

    pred_z = np.concatenate(pred_z_list, axis=0)
    true_z = np.concatenate(true_z_list, axis=0)
    mask = np.concatenate(mask_list, axis=0)

    pred = pred_z * label_std + label_mean
    true = true_z * label_std + label_mean

    metrics: List[TargetMetrics] = []
    for j, name in enumerate(label_cols):
        valid = mask[:, j] > 0
        if valid.sum() < 2:
            continue
        p = pred[valid, j]
        t = true[valid, j]
        mae = float(np.mean(np.abs(p - t)))
        corr = float(np.corrcoef(p, t)[0, 1]) if len(p) > 1 else 0.0
        metrics.append(TargetMetrics(name=name, mae=mae, corr=corr, n=int(valid.sum())))
    return metrics

def print_target_metrics(metrics: Sequence[TargetMetrics], title: str = "") -> None:
    """Pretty-print per-target evaluation lines."""
    if title:
        print(title)
    for m in metrics:
        print(f"  {m.name:28s}  MAE={m.mae:.5f}  corr={m.corr:+.3f}  n={m.n}")

def print_return_metrics(metrics: Sequence[TargetMetrics]) -> None:
    """Highlight future_return correlation (primary signal quality check)."""
    by_name = {m.name: m for m in metrics}
    m = by_name.get(RETURN_COL)
    if m is None:
        print(f"  {RETURN_COL:28s}  (no valid samples)")
        return
    print(f"Primary return correlation:")
    print(f"  {m.name:28s}  corr={m.corr:+.3f}  MAE={m.mae:.5f}  n={m.n}")

def evaluate_regime_head(
    model: MarketTransformer,
    loader: DataLoader,
    *,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray]:
    """Collect regime predictions and ground truth for sklearn reports."""
    model.eval()
    pred_list: List[int] = []
    true_list: List[int] = []
    with torch.no_grad():
        for xb, _yb_z, _mb, rb in loader:
            xb = xb.to(device)
            _, regime_logits = model(xb)
            pred_list.extend(torch.argmax(regime_logits, dim=1).cpu().numpy().tolist())
            true_list.extend(rb.numpy().tolist())
    return np.array(pred_list), np.array(true_list)

def evaluate_regime_accuracy(
    model: MarketTransformer,
    loader: DataLoader,
    *,
    device: torch.device,
) -> float:
    """Test-set accuracy for the volatility-regime classification head."""
    pred, true = evaluate_regime_head(model, loader, device=device)
    if len(true) == 0:
        return 0.0
    return float((pred == true).mean())

@dataclass
class ExportQualityAssessment:
    """Tiered export quality result (sanity vs promotion-ready)."""

    tier: str
    checks: Dict[str, Any]
    warnings: List[str]
    disclaimer: str

def assess_export_quality(
    metrics: Sequence[TargetMetrics],
    *,
    resolution: str,
    min_return_corr: float,
    regime_accuracy: float | None = None,
) -> ExportQualityAssessment:
    """Evaluate sanity and promotion tiers without blocking on promotion failures."""
    res = resolution.strip().lower()
    by_name = {m.name: m for m in metrics}
    return_m = by_name.get(RETURN_COL)
    vol_m = by_name.get("future_volatility")
    checks: Dict[str, Any] = {}
    warnings: List[str] = []

    if return_m is None:
        return ExportQualityAssessment(
            tier="blocked",
            checks=checks,
            warnings=["no valid future_return test samples"],
            disclaimer=EXPORT_QUALITY_DISCLAIMER,
        )

    sanity_floor = float(min_return_corr)
    promotion_return_floor = float(PROMOTION_RETURN_CORR.get(res, 0.06))
    checks["future_return"] = {
        "corr": return_m.corr,
        "sanity_floor": sanity_floor,
        "promotion_floor": promotion_return_floor,
        "sanity_passed": return_m.corr >= sanity_floor,
        "promotion_passed": return_m.corr >= promotion_return_floor,
    }
    if not checks["future_return"]["sanity_passed"]:
        return ExportQualityAssessment(
            tier="blocked",
            checks=checks,
            warnings=[
                f"future_return corr {return_m.corr:.4f} < sanity floor {sanity_floor:.4f}"
            ],
            disclaimer=EXPORT_QUALITY_DISCLAIMER,
        )

    if vol_m is not None:
        checks["future_volatility"] = {
            "corr": vol_m.corr,
            "promotion_floor": PROMOTION_VOL_CORR,
            "promotion_passed": vol_m.corr >= PROMOTION_VOL_CORR,
        }
        if not checks["future_volatility"]["promotion_passed"]:
            warnings.append(
                f"promotion: future_volatility corr {vol_m.corr:.4f} "
                f"< {PROMOTION_VOL_CORR:.4f}"
            )

    if regime_accuracy is not None:
        checks["regime_accuracy"] = {
            "value": float(regime_accuracy),
            "promotion_floor": PROMOTION_REGIME_ACCURACY,
            "promotion_passed": float(regime_accuracy) >= PROMOTION_REGIME_ACCURACY,
        }
        if not checks["regime_accuracy"]["promotion_passed"]:
            warnings.append(
                f"promotion: regime accuracy {regime_accuracy:.4f} "
                f"< {PROMOTION_REGIME_ACCURACY:.4f}"
            )

    if not checks["future_return"]["promotion_passed"]:
        warnings.append(
            f"promotion: future_return corr {return_m.corr:.4f} "
            f"< {promotion_return_floor:.4f}"
        )

    promotion_ready = (
        checks["future_return"]["promotion_passed"]
        and (vol_m is None or checks.get("future_volatility", {}).get("promotion_passed", False))
        and (
            regime_accuracy is None
            or checks.get("regime_accuracy", {}).get("promotion_passed", False)
        )
    )
    tier = "promotion_ready" if promotion_ready else "sanity_pass"
    return ExportQualityAssessment(
        tier=tier,
        checks=checks,
        warnings=warnings,
        disclaimer=EXPORT_QUALITY_DISCLAIMER,
    )

def check_export_quality_gate(
    metrics: Sequence[TargetMetrics],
    min_return_corr: float,
) -> None:
    """Raise if future_return test correlation is below export floor."""
    assessment = assess_export_quality(
        metrics,
        resolution="15m",
        min_return_corr=min_return_corr,
    )
    if assessment.tier == "blocked":
        msg = assessment.warnings[0] if assessment.warnings else "export quality blocked"
        raise RuntimeError(f"Export blocked: {msg}")

def export_transformer_bundle(
    model: MarketTransformer,
    export_dir: Path,
    *,
    device: torch.device,
    window_len: int,
    n_features: int,
    feature_cols: Sequence[str],
    label_mean: np.ndarray,
    label_std: np.ndarray,
    q_edges: np.ndarray,
    config: Dict[str, Any],
    test_metrics: Sequence[TargetMetrics] | None = None,
    regime_accuracy: float | None = None,
    verify: bool = True,
    enforce_quality_gate: bool = True,
) -> Tuple[Path, Path, Path]:
    """Export ONNX + feature_config.json + metadata_transformer.json."""
    resolution = str(config.get("resolution") or "15m")
    min_corr = float(config.get("min_export_return_corr") or 0.02)
    export_quality: Dict[str, Any] = {}
    if test_metrics is not None:
        assessment = assess_export_quality(
            test_metrics,
            resolution=resolution,
            min_return_corr=min_corr,
            regime_accuracy=regime_accuracy,
        )
        export_quality = {
            "tier": assessment.tier,
            "checks": assessment.checks,
            "warnings": assessment.warnings,
            "disclaimer": assessment.disclaimer,
        }
        if assessment.warnings:
            for warning in assessment.warnings:
                print(f"Export quality warning: {warning}")
        print(f"Export quality tier: {assessment.tier}")
        if enforce_quality_gate and assessment.tier == "blocked":
            msg = assessment.warnings[0] if assessment.warnings else "quality gate failed"
            raise RuntimeError(f"Export blocked: {msg}")

    export_dir.mkdir(parents=True, exist_ok=True)
    onnx_name = onnx_filename_for_resolution(resolution)
    onnx_path = export_dir / onnx_name
    cfg_path = export_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME
    meta_path = export_dir / TRANSFORMER_METADATA_FILENAME

    dummy = torch.randn(1, window_len, n_features, device=device)
    export_kwargs: Dict[str, Any] = {
        "input_names": ["window"],
        "output_names": ["continuous_pred", "regime_logits"],
        "dynamic_axes": {
            "window": {0: "batch"},
            "continuous_pred": {0: "batch"},
            "regime_logits": {0: "batch"},
        },
        "opset_version": 17,
        "export_params": True,
    }
    try:
        torch.onnx.export(
            model,
            dummy,
            str(onnx_path),
            dynamo=False,
            **export_kwargs,
        )
    except TypeError:
        torch.onnx.export(model, dummy, str(onnx_path), **export_kwargs)

    if verify:
        import onnx as onnx_lib
        import onnxruntime as ort

        check_model = onnx_lib.load(str(onnx_path), load_external_data=False)
        external = [
            init.name for init in check_model.graph.initializer if init.data_location == 1
        ]
        if external:
            raise RuntimeError(
                f"ONNX export left {len(external)} weights external "
                f"(e.g. {external[0]}). Model will not run standalone."
            )

        sess = ort.InferenceSession(str(onnx_path))
        onnx_cont, onnx_reg = sess.run(None, {"window": dummy.cpu().numpy()})
        with torch.no_grad():
            torch_cont, torch_reg = model(dummy)
        diff_cont = np.abs(onnx_cont - torch_cont.cpu().numpy()).max()
        diff_reg = np.abs(onnx_reg - torch_reg.cpu().numpy()).max()
        print(f"ONNX verify continuous_pred max diff: {diff_cont:.2e}")
        print(f"ONNX verify regime_logits max diff: {diff_reg:.2e}")
        print(f"ONNX output shapes: {onnx_cont.shape}, {onnx_reg.shape}")

    metrics_dict = {}
    if test_metrics:
        metrics_dict = {m.name: {"mae": m.mae, "corr": m.corr, "n": m.n} for m in test_metrics}

    feature_config = feature_config_from_training_export(
        feature_cols=feature_cols,
        window_len=window_len,
        label_mean=label_mean,
        label_std=label_std,
        q_edges=q_edges,
        config=config,
    )
    cfg_path.write_text(json.dumps(feature_config, indent=2), encoding="utf-8")

    metadata = metadata_from_training_export(
        resolution=resolution,
        label_mean=label_mean,
        label_std=label_std,
        config=config,
        test_metrics=metrics_dict,
        export_quality=export_quality or None,
    )
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    return onnx_path, cfg_path, meta_path


## Training runner

In [ ]:
"""Run per-TF transformer training from CLI or Colab."""

from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any, Sequence

import pandas as pd
import torch
from torch.utils.data import DataLoader

def run_training(
    *,
    resolution: str,
    export_dir: Path,
    epochs: int | None = None,
    history_days: int | None = None,
    raw_cache_path: Path | None = None,
    refresh_data: bool = False,
    enforce_quality_gate: bool = True,
) -> None:
    """Train and export a single per-TF transformer bundle."""
    res = resolution.strip().lower()
    if res not in SUPPORTED_RESOLUTIONS:
        raise ValueError(f"Unsupported resolution {resolution!r}; use one of {SUPPORTED_RESOLUTIONS}")

    config = dict(default_training_config(res))
    if epochs is not None:
        config["epochs"] = epochs
    if history_days is not None:
        config["history_days"] = history_days

    set_training_seed(config["seed"])
    print(json.dumps(config, indent=2))

    cache = raw_cache_path or Path(f"btcusd_{res}_raw.parquet")
    if cache.is_file() and not refresh_data:
        print(f"Loading cached raw data from {cache}")
        raw_df = pd.read_parquet(cache)
        validate_derivatives_coverage(
            raw_df,
            min_coverage=config["min_derivatives_coverage"],
            warn_coverage=config["derivatives_coverage_warn"],
        )
    else:
        raw_df = fetch_history_bundle(
            symbol=config["symbol"],
            resolution=config["resolution"],
            history_days=config["history_days"],
            base_url=config["base_url"],
            min_derivatives_coverage=config["min_derivatives_coverage"],
            derivatives_coverage_warn=config["derivatives_coverage_warn"],
        )
        cache.parent.mkdir(parents=True, exist_ok=True)
        raw_df.to_parquet(cache)
        print(f"Cached raw pull to {cache}")

    resolution_minutes = int(config["resolution_minutes"])
    feat_df = add_features(
        raw_df,
        resolution_minutes=resolution_minutes,
        atr_period=config["atr_period"],
    ).dropna().reset_index(drop=True)
    feat_df = compute_market_labels(
        feat_df,
        return_horizon_bars=int(config["return_horizon_bars"]),
        path_label_horizon_bars=config["path_label_horizon_bars"],
        mae_floor_atr_mult=config["mae_floor_atr_mult"],
    )
    feat_df = trim_label_tail(
        feat_df,
        return_horizon_bars=int(config["return_horizon_bars"]),
        path_label_horizon_bars=config["path_label_horizon_bars"],
    )

    print(f"raw bars: {len(raw_df)}, feature rows: {len(feat_df)}")
    nan_rates = label_nan_summary(feat_df)
    print("label NaN rates:", nan_rates)

    x_all, y_all = build_windows(
        feat_df,
        FEATURE_COLS,
        CONTINUOUS_LABEL_COLS,
        config["window_len"],
        config["stride"],
    )
    splits = split_purged_windows(
        x_all,
        y_all,
        train_frac=config["train_frac"],
        val_frac=config["val_frac"],
        embargo_bars=config["embargo_bars"],
    )

    label_mean, label_std = fit_label_stats(splits["y_train"])
    y_train_z, m_train = standardize_labels(splits["y_train"], label_mean, label_std)
    y_val_z, m_val = standardize_labels(splits["y_val"], label_mean, label_std)
    y_test_z, m_test = standardize_labels(splits["y_test"], label_mean, label_std)

    q_edges = fit_vol_regime_edges(splits["y_train"], config["vol_regime_quantiles"])
    r_train = to_vol_regime(splits["y_train"], q_edges)
    r_val = to_vol_regime(splits["y_val"], q_edges)
    r_test = to_vol_regime(splits["y_test"], q_edges)

    train_loader = DataLoader(
        WindowDataset(splits["x_train"], y_train_z, m_train, r_train),
        batch_size=config["batch_size"],
        shuffle=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        WindowDataset(splits["x_val"], y_val_z, m_val, r_val),
        batch_size=config["batch_size"],
        shuffle=False,
    )
    test_loader = DataLoader(
        WindowDataset(splits["x_test"], y_test_z, m_test, r_test),
        batch_size=config["batch_size"],
        shuffle=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    model = MarketTransformer(
        n_features=len(FEATURE_COLS),
        d_model=config["d_model"],
        nhead=config["nhead"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        max_len=config["window_len"],
        n_continuous=len(CONTINUOUS_LABEL_COLS),
    ).to(device)

    result = train_transformer(model, train_loader, val_loader, config, device=device)
    model.load_state_dict(result.model_state)
    model.eval()

    test_metrics = evaluate_continuous_targets(
        model,
        test_loader,
        device=device,
        label_mean=label_mean,
        label_std=label_std,
    )
    print_return_metrics(test_metrics)
    print_target_metrics(test_metrics, title="Test set — all continuous targets:")

    regime_accuracy = evaluate_regime_accuracy(model, test_loader, device=device)
    print(f"Regime head test accuracy: {regime_accuracy:.3f}")

    export_dir.mkdir(parents=True, exist_ok=True)
    onnx_path, cfg_path, meta_path = export_transformer_bundle(
        model,
        export_dir,
        device=device,
        window_len=config["window_len"],
        n_features=len(FEATURE_COLS),
        feature_cols=FEATURE_COLS,
        label_mean=label_mean,
        label_std=label_std,
        q_edges=q_edges,
        config=config,
        test_metrics=test_metrics,
        regime_accuracy=regime_accuracy,
        enforce_quality_gate=enforce_quality_gate,
    )
    print(f"Exported {onnx_path}")
    print(f"Exported {cfg_path}")
    print(f"Exported {meta_path}")

def run_all_training(
    *,
    resolutions: Sequence[str] | None = None,
    export_dir: Path,
    cache_dir: Path | None = None,
    epochs: int | None = None,
    history_days: int | None = None,
    refresh_data: bool = False,
    enforce_quality_gate: bool = True,
    continue_on_error: bool = False,
) -> list[dict[str, Any]]:
    """Train and export all requested per-TF transformer bundles."""
    tfs = list(resolutions or SUPPORTED_RESOLUTIONS)
    results: list[dict[str, Any]] = []
    cache_root = cache_dir or Path(".")

    for resolution in tfs:
        res = resolution.strip().lower()
        tf_export_dir = export_dir / bundle_dir_name(res)
        raw_cache_path = cache_root / f"btcusd_{res}_raw.parquet"
        print(f"\n{'=' * 60}\nTraining {res} -> {tf_export_dir}\n{'=' * 60}")

        try:
            run_training(
                resolution=res,
                export_dir=tf_export_dir,
                epochs=epochs,
                history_days=history_days,
                raw_cache_path=raw_cache_path,
                refresh_data=refresh_data,
                enforce_quality_gate=enforce_quality_gate,
            )
            results.append(
                {
                    "resolution": res,
                    "status": "ok",
                    "export_dir": str(tf_export_dir),
                    "error": None,
                }
            )
        except Exception as exc:
            results.append(
                {
                    "resolution": res,
                    "status": "failed",
                    "export_dir": str(tf_export_dir),
                    "error": str(exc),
                }
            )
            if not continue_on_error:
                raise

    return results

def main() -> None:
    parser = argparse.ArgumentParser(description="Train per-TF BTCUSD transformer")
    parser.add_argument("--resolution", choices=list(SUPPORTED_RESOLUTIONS), default=None)
    parser.add_argument("--all", action="store_true", help="Train all supported resolutions")
    parser.add_argument("--export-dir", type=Path, default=Path("export"))
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--history-days", type=int, default=None)
    parser.add_argument("--raw-cache", type=Path, default=None)
    parser.add_argument("--refresh-data", action="store_true")
    parser.add_argument("--skip-quality-gate", action="store_true")
    parser.add_argument(
        "--continue-on-error",
        action="store_true",
        help="When using --all, continue training remaining TFs after a failure",
    )
    args = parser.parse_args()

    if args.all and args.resolution:
        parser.error("Use either --resolution or --all, not both")
    if not args.all and not args.resolution:
        parser.error("Specify --resolution <tf> or --all")

    enforce_quality_gate = not args.skip_quality_gate
    if args.all:
        run_all_training(
            export_dir=args.export_dir,
            epochs=args.epochs,
            history_days=args.history_days,
            refresh_data=args.refresh_data,
            enforce_quality_gate=enforce_quality_gate,
            continue_on_error=args.continue_on_error,
        )
        return

    run_training(
        resolution=args.resolution,
        export_dir=args.export_dir,
        epochs=args.epochs,
        history_days=args.history_days,
        raw_cache_path=args.raw_cache,
        refresh_data=args.refresh_data,
        enforce_quality_gate=enforce_quality_gate,
    )

if __name__ == "__main__":
    main()


## Configure and train

In [ ]:
import json

print("Per-TF label horizons (~8h wall-clock):")
for res in SUPPORTED_RESOLUTIONS:
    bars = label_horizon_bars_for_resolution(RESOLUTION_MINUTES[res])
    print(f"  {res:>4s}: {bars:3d} bars")

print("\n15m default_training_config:")
print(json.dumps(default_training_config("15m"), indent=2))


In [ ]:
from pathlib import Path

# Train all TFs or a subset, e.g. ["15m"] for a quick smoke test.
resolutions = list(SUPPORTED_RESOLUTIONS)
export_dir = Path("/content/export")
export_dir.mkdir(parents=True, exist_ok=True)
cache_dir = Path("/content/cache")
cache_dir.mkdir(parents=True, exist_ok=True)

# Optional overrides (None = use per-TF defaults from default_training_config).
epochs = None  # e.g. 5 for smoke test
history_days = None  # e.g. 900; BTCUSD India history ~950 days as of 2026
refresh_data = False  # set True to re-fetch from Delta API instead of parquet cache
continue_on_error = True  # finish remaining TFs if one fails quality gate
enforce_quality_gate = True  # set False to export even if future_return corr is low


## Train all resolutions

In [ ]:
results = run_all_training(
    resolutions=resolutions,
    export_dir=export_dir,
    cache_dir=cache_dir,
    epochs=epochs,
    history_days=history_days,
    refresh_data=refresh_data,
    continue_on_error=continue_on_error,
    enforce_quality_gate=enforce_quality_gate,
)

for result in results:
    print(result)


## Results summary

In [ ]:
import json
from pathlib import Path

if "results" not in globals():
    raise NameError("Run the training cell above first.")

print(f"{'Resolution':<10}{'Status':<8}{'Return Corr':<14}Export Dir")
for result in results:
    corr_str = "-"
    if result["status"] == "ok":
        meta_path = Path(result["export_dir"]) / "metadata_transformer.json"
        if meta_path.is_file():
            metrics = json.loads(meta_path.read_text()).get("test_metrics", {})
            corr = metrics.get("future_return", {}).get("corr")
            if corr is not None:
                corr_str = f"{corr:+.4f}"
    print(
        f"{result['resolution']:<10}{result['status']:<8}{corr_str:<14}{result['export_dir']}"
    )
    if result["status"] != "ok" and result.get("error"):
        print(f"  error: {result['error']}")


## Download exports

In [ ]:
import shutil
from pathlib import Path

if "export_dir" not in globals():
    raise NameError("Run the config cell above first.")

if not export_dir.is_dir():
    raise FileNotFoundError(f"Export dir not found: {export_dir}. Run training first.")

bundles = [p for p in export_dir.iterdir() if p.is_dir()]
if not bundles:
    print("No TF bundles exported. Check training results above.")
else:
    zip_path = shutil.make_archive("/content/transformer_exports", "zip", export_dir)
    print(f"Download: {zip_path} ({len(bundles)} bundle(s))")

    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print("Not running in Colab; download manually from the path above.")


## Notes before wiring into your live agent

- **No trading label was trained** — models predict continuous market properties + a volatility-regime
  class. Trading decisions are a downstream step in JackSparrow.
- **Label horizon** scales to ~8 hours wall-clock per TF (32 bars on 15m, 96 on 5m, etc.). It is
  independent of `window_len` (input lookback) and worth sweeping separately.
- **Early stopping is enabled** by default (patience 12, max 120 epochs); best val-loss checkpoint is
  used for export.
- **Feature parity is the #1 deployment failure mode** — train/serve uses
  `feature_store/transformer_btcusd/` (not `unified_feature_engine`). Run
  `pytest tests/unit/test_transformer_btcusd_feature_parity.py` before deploy.
- **Export quality tiers** — sanity floors block broken exports; promotion targets in metadata are
  informational, not proof of tradability.
- **ONNX export** embeds all weights in a single file (`dynamo=False`) and is verified before download.